<a href="https://colab.research.google.com/github/Psam4ord/Terraform-cloud-turorial-repo/blob/dev/Physics_informed_Neural_networks_for_complex_images.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# 1. Import Libraries
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
import torchvision.transforms as transforms

import matplotlib.pyplot as plt

In [ ]:
# ============================================================
# 2. Device Configuration
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cpu


In [ ]:
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

print("Updated training transformations with RandomHorizontalFlip and RandomCrop.")

Updated training transformations with RandomHorizontalFlip and RandomCrop.


In [ ]:
# ============================================================
# 4. Load CIFAR-10 Dataset
# ============================================================

train_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transform_train
)

test_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transform_test
)

train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

classes = (
'plane','car','bird','cat','deer',
'dog','frog','horse','ship','truck'
)

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)

        # Define dropout layers
        self.dropout_conv = nn.Dropout(0.25)
        self.dropout_fc = nn.Dropout(0.5)

        self.fc1 = nn.Linear(64 * 8 * 8, 512)
        self.fc2 = nn.Linear(512, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        # First block
        x = self.relu(self.conv1(x))
        x = self.pool(x)
        x = self.dropout_conv(x)

        # Second block
        features = self.relu(self.conv2(x))
        x = self.pool(features)
        x = self.dropout_conv(x)

        # Fully connected layers
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        x = self.dropout_fc(x)
        x = self.fc2(x)

        return x, features

In [ ]:
# ============================================================
# 6. PINN-Inspired Loss Functions
# ============================================================

def total_variation_loss(x):

    tv_h = torch.mean(torch.abs(x[:,:,1:,:] - x[:,:,:-1,:]))
    tv_w = torch.mean(torch.abs(x[:,:,:,1:] - x[:,:,:,:-1]))

    return tv_h + tv_w


def gradient_regularization(x):

    grad_x = x[:,:,1:,:] - x[:,:,:-1,:]
    grad_y = x[:,:,:,1:] - x[:,:,:,:-1]

    grad_penalty = torch.mean(grad_x**2) + torch.mean(grad_y**2)

    return grad_penalty

    # ============================================================
# 7. Initialize Model, Loss, Optimizer
# ============================================================

model = SimpleCNN().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

# PINN constraint weights
lambda_tv = 0.001
lambda_grad = 0.001



In [ ]:
import torch.optim.lr_scheduler

num_epochs = 100

# Instantiate a learning rate scheduler
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

for epoch in range(num_epochs):
    running_loss = 0.0
    model.train() # Set to training mode
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs, features = model(images);
        ce_loss = criterion(outputs, labels);
        tv_loss = total_variation_loss(features);
        grad_loss = gradient_regularization(features);

        loss = ce_loss + lambda_tv * tv_loss + lambda_grad * grad_loss
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    # Step the scheduler at the end of each epoch
    scheduler.step()

    print(f"Epoch [{epoch+1}/{num_epochs}] Loss: {running_loss/len(train_loader):.4f}, Current LR: {scheduler.get_last_lr()[0]:.6f}")

In [8]:
correct = 0
total = 0

model.eval() # Set model to evaluation mode
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs, _ = model(images)
        _, predicted = torch.max(outputs.data, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Test Accuracy after improvements: {accuracy:.2f}%")

Test Accuracy after improvements: 74.29%


In [ ]:
#Visualization After Testing

import numpy as np

# Get a batch of test images and labels
dataiter = iter(test_loader)
images, labels = next(dataiter)

# Move images and labels to the device
images = images.to(device)
labels = labels.to(device)

# Make predictions
outputs, _ = model(images)
_, predicted = torch.max(outputs, 1)

# Convert images to numpy for plotting
# Unnormalize the images for better visualization
img_np = images.cpu().numpy()
img_np = img_np / 2 + 0.5  # unnormalize

# Plotting
fig = plt.figure(figsize=(15, 8))
for i in range(12):
    ax = fig.add_subplot(2, 6, i + 1, xticks=[], yticks=[])
    # Transpose dimensions from (C, H, W) to (H, W, C) for matplotlib
    ax.imshow(np.transpose(img_np[i], (1, 2, 0)))
    ax.set_title(f"True: {classes[labels[i]]}\nPred: {classes[predicted[i]]}",
                 color=("green" if predicted[i] == labels[i] else "red"))
plt.tight_layout()
plt.show()